In [1]:
import torch
from torch.utils.data import DataLoader, TensorDataset
import pandas as pd
import numpy as np
import os
from dragonnet import DragonNetBase  # 假设这个类定义在 dragonnet.py 中
from model_utils import *  # 假设你有自定义 loss 在这个文件中

In [2]:
df = pd.read_csv('20w_instance.csv')  # 请替换成你的 CSV 数据文件路径
df.replace({np.inf: 99999.0, -np.inf: -99999.0})
df.fillna(0, inplace=True)

In [3]:
pd.set_option('display.max_columns', None)  # 显示所有列
pd.set_option('display.width', None)  
df.head(20)

,timestamp,open,high,low,close,volume,close_time,quote_asset_volume,number_of_trades,taker_buy_base_vol,taker_buy_quote_vol,ignore,vwap,returns,alpha006_60,alpha006_240,alpha007_60,alpha007_240,alpha009_60,alpha009_240,alpha010_60,alpha010_240,alpha012_60,alpha012_240,alpha013_60,alpha013_240,alpha018_60,alpha018_240,alpha021_60,alpha021_240,alpha023_60,alpha023_240,alpha024_60,alpha024_240,alpha026_60,alpha026_240,alpha028_60,alpha028_240,alpha032_60,alpha032_240,alpha035_60,alpha035_240,alpha041,alpha043_60,alpha043_240,alpha046_60,alpha046_240,alpha049_60,alpha049_240,alpha051_60,alpha051_240,alpha053_60,alpha053_240,alpha054,alpha060,alpha084_60,alpha084_240,alpha101,alpha_102_60,alpha_102_240,alpha_103_60,alpha_103_240,alpha_104_60,alpha_104_240,alpha_105_60,alpha_105_240,alpha_106,alpha_107_60,alpha_107_240,alpha_108_60,alpha_108_240,alpha_109_60,alpha_109_240,alpha_110_60,alpha_110_240,alpha_111_60,alpha_111_240,alpha_112_20,alpha_112_60,alpha_112_240,alpha_113_60,alpha_113_240,alpha_115_60,alpha_115_240,alpha_116_60,alpha_116_240,alpha_117_60,alpha_117_240,alpha_118_60,alpha_118_240,alpha_119,alpha_120_60,alpha_120_240,alpha_121,alpha_122_60,alpha_122_240,alpha_123_60,alpha_123_240,alpha_124,alpha_125_60,alpha_125_240,alpha_126_60,alpha_126_240,alpha_127_60,alpha_127_240,alpha_128_60,alpha_128_240,alpha_129_60,alpha_129_240,alpha_130_60,alpha_130_240,alpha_131_60,alpha_131_240,treatment,amplitude,custom_std
0,2018-01-08 00:45:00,-0.601162,-0.601370,-0.600846,-0.601198,-0.401647,1515372359999,-0.451864,-0.589265,-0.382723,-0.423655,0,-0.601131,-0.058938,-0.478367,0.557544,0.569417,1.340576,0.549834,-0.308107,0.549824,-0.308351,0.521648,-0.343004,-0.645228,0.469398,-0.705966,-0.522179,-1.160230,-1.169886,0.198089,0.208236,0.241377,0.649786,0.472322,-0.515222,0.149187,0.118500,0.0,0.0,-0.208968,-0.504323,0.106818,-0.837655,1.041370,-0.576319,-0.680546,0.054357,0.038982,0.054269,0.038932,3.413111e-01,0.340752,1.409367,0.152433,0.0,0.0,-0.325542,-0.803803,0.473173,-1.175197,0.696682,-0.053164,-0.029713,-0.822993,0.515035,-0.311870,-1.150360,-1.200964,1.356252,0.993599,-0.819741,0.524545,0.532303,-0.523581,0.326768,0.226351,-0.776389,-0.321967,-0.687830,-0.011255,-0.013696,-0.655699,-0.803124,0.379374,0.225880,-0.000515,-0.518488,-0.550089,0.296393,-0.428870,0.402128,1.258107,2.127393,0.0,0.0,0.706984,0.982409,-0.216883,0.0,-0.518488,-0.691398,0.047240,-0.435758,-0.443433,0.650347,0.977645,1.041809,0.736140,1.138374,0.788826,-0.500055,0.094926,0,0.023780,0.009340
1,2018-01-08 00:46:00,-0.601163,-0.601510,-0.601290,-0.601642,-0.399952,1515372419999,-0.450876,-0.590714,-0.320208,-0.387070,0,-0.601320,-0.735798,-0.555990,0.603822,0.653159,1.319233,0.500676,-0.273159,0.500666,-0.273403,0.472478,-0.308049,-0.685194,0.496627,0.121913,0.620768,-1.160230,-1.169886,0.198089,0.208236,0.251980,0.635752,0.472322,-0.515222,0.373336,0.341945,0.0,0.0,-0.379087,-0.569527,-0.368992,-0.947052,1.097184,-0.576319,-0.680546,0.519887,0.512080,0.519967,0.512106,-1.200687e+00,0.934767,1.409367,0.398156,0.0,0.0,-1.598171,-0.741283,0.415918,-1.071453,0.618481,-0.053164,-0.029713,-0.756061,0.452508,-0.224163,-1.287710,-1.271942,1.344411,0.994830,-0.752259,0.462551,0.583642,-0.569260,0.313558,0.234215,-0.849433,-0.431055,-0.768292,-0.203304,-0.231024,-0.655972,-0.802947,0.243474,0.159279,-0.000518,-0.593657,-0.500969,0.273465,-0.856904,0.410814,1.277793,2.127374,0.0,0.0,0.708238,0.983954,-1.025779,0.0,-0.593657,-0.740199,-0.038514,-0.435758,-0.443433,0.395837,0.931820,1.017199,0.726799,1.117587,0.781250,0.174734,-0.745832,0,0.021177,0.009623
2,2018-01-08 00:47:00,-0.601672,-0.601750,-0.601648,-0.601527,-0.405586,1515372479999,-0.454314,-0.578157,-0.412709,-0.441280,0,-0.601870,0.189746,-0.617252,0.673922,0.611288,1.255203,0.420421,-0.327548,0.420411,-0.327792,0.392202,-0.362449,-0.722623,0.528612,-0.878255,-0.730424,-1.160230,-1.169886,0.198089,0.208236,0.249229,0.634779,0.472322,-0.515222,-0.230751,-0.262287,0.0,0.0,0.032301,-0.440888,0.798963,-0.9474

In [4]:
# === 提取特征和标签 ===
# 你需要确认这些列是否与你的数据匹配
features = df[[col for col in df.columns if col not in ['timestamp', 'close_time', 'treatment', 'amplitude', 'custom_std']]].values.astype(np.float32)
treatment = df['treatment'].values.astype(np.float32)  # 二值处理变量 (0/1)
amplitude = df['amplitude'].values.astype(np.float32)  # shape: (N, k)
custom_std = df['custom_std'].values.astype(np.float32)  # shape: (N, k)

# === 构建 Dataset 和 DataLoader ===
dataset = TensorDataset(torch.from_numpy(features), torch.from_numpy(treatment), 1000*torch.from_numpy(amplitude), 1000*torch.from_numpy(custom_std))
dataloader = DataLoader(dataset, batch_size=1024, shuffle=True, num_workers=16)

/Users/bytedance/miniconda3/lib/python3.11/site-packages/torch/utils/data/dataloader.py:617: UserWarning: This DataLoader will create 16 worker processes in total. Our suggested max number of worker in current system is 10 (`cpuset` is not taken into account), which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


In [5]:
from torch.optim.lr_scheduler import StepLR


def main():
  # === 初始化模型 ===
  input_dim = features.shape[1]
  model = DragonNetBase()
  dummy_input = torch.randn(16, input_dim)  # 这里的维度必须和实际训练时一致
  model(dummy_input)
  optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
  scheduler = StepLR(optimizer, step_size=1, gamma=0.8)
  model.train()

#   opt_return_trmt = torch.optim.Adam(list(model.trmt_return_1h_tower.parameters()) + list(model.bottom_tower.parameters()), lr=1e-3)
#   opt_return_ctrl = torch.optim.Adam(list(model.ctrl_return_1h_tower.parameters()) + list(model.bottom_tower.parameters()), lr=1e-3)
#   opt_sigma_trmt = torch.optim.Adam(list(model.trmt_sigma_1h_tower.parameters()) + list(model.bottom_tower.parameters()), lr=1e-3)
#   opt_sigma_ctrl = torch.optim.Adam(list(model.ctrl_sigma_1h_tower.parameters()) + list(model.bottom_tower.parameters()), lr=1e-3)
#   opt_return_tar = torch.optim.Adam(list(model.trmt_return_1h_tower.parameters()) + list(model.ctrl_return_1h_tower.parameters() + list(model.bottom_tower.parameters())), lr=1e-3)
#   opt_sigma_tar = torch.optim.Adam(list(model.trmt_sigma_1h_tower.parameters()) + list(model.ctrl_sigma_1h_tower.parameters() + list(model.bottom_tower.parameters())), lr=1e-3)
#   opt_t = torch.optim.Adam(list(model.t_tower.parameters()) + list(model.bottom_tower.parameters()), lr=1e-3)
  

  # === 训练循环 ===
  for epoch in range(20):
      for step, (x, t, y0, y1) in enumerate(dataloader):
          mask_t0 = (t == 0).float()
          mask_t1 = (t == 1).float()
          # print('mask_t1.shape',mask_t1.shape)
          # print('mask_t0.shape',mask_t0.shape)
          pred_y0_ctrl, pred_y0_trmt, pred_y1_ctrl, pred_y1_trmt, pred_t, return_eps, sigma_eps = model(x)  # DragonNet 输出三个分支
          return_pred_ctrl, return_loss_ctrl = get_pred_and_loss(pred_y0_ctrl, y0, 1, 'mse', mask_t0)
          return_pred_trmt, return_loss_trmt = get_pred_and_loss(pred_y0_trmt, y0, 1, 'mse', mask_t1)
          sigma_pred_ctrl, sigma_loss_ctrl = get_pred_and_loss(pred_y1_ctrl, y1, 1, 'mse', mask_t0)
          sigma_pred_trmt, sigma_loss_trmt = get_pred_and_loss(pred_y1_trmt, y1, 1, 'mse', mask_t1)
          t_pred, t_loss = get_pred_and_loss(pred_t, t, 1, 'logloss', None)
          _, return_tar_loss = get_tarreg_pred_and_loss('return_tar_loss',[t_pred, pred_y0_ctrl, pred_y0_trmt], [t, y0], 1, return_eps)
          _, sigma_tar_loss = get_tarreg_pred_and_loss('sigma_tar_loss', [t_pred, pred_y1_ctrl, pred_y1_trmt], [t, y1], 1, sigma_eps)
          loss = return_loss_ctrl + return_loss_trmt + t_loss + sigma_loss_ctrl + sigma_loss_trmt + return_tar_loss + sigma_tar_loss
          optimizer.zero_grad()
          loss.backward()
          #total_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2000)
          #print("Grad norm:", total_norm.item())
          optimizer.step()
          # print('return_pred_ctrl.shape', return_pred_ctrl.shape)
          # print('return_pred_ctrl', return_pred_ctrl)
          # print('return_loss_ctrl',return_loss_ctrl)
          
          # print('mask_t1',mask_t1)
          # print('mask_t0',mask_t0)
          # print('y0', y0)
          # print('y0*mask_t1', y0*mask_t1)
          # print('y0*mask_t0', y0*mask_t0)
          # print('torch.sum(y0*mask_t1)', torch.sum(y0*mask_t1))
          # print('torch.sum(y0*mask_t0)', torch.sum(y0*mask_t0))
          # print('torch.sum(mask_t1)', torch.sum(mask_t1))
          # print('torch.sum(mask_t0)', torch.sum(mask_t0))
      
      print(f"----------------Step {epoch+1}-----------------")
      print(f"Return Trmt Loss: {return_loss_trmt:.4f}")
      print(f"Return Ctrl Loss: {return_loss_ctrl:.4f}")
      print(f"Sigma Trmt Loss: {sigma_loss_trmt:.4f}")
      print(f"Sigma Ctrl Loss: {sigma_loss_ctrl:.4f}")
      print(f"t Loss: {t_loss:.4f}")
      print(f"return tar Loss: {return_tar_loss:.4f}")
      print(f"sigma tar Loss: {sigma_tar_loss:.4f}")
      print(f"Pred_t: {torch.mean(t_pred)}")
      print(f"Label_t: {torch.mean(t)}")
      print(f"Pred_return_trmt: {torch.sum(return_pred_trmt.squeeze()*mask_t1)/torch.sum(mask_t1.float())}")
      print(f"Label_return_trmt: {torch.sum(y0.squeeze()*mask_t1.float())/torch.sum(mask_t1.float())}")
      print(f"Pred_return_ctrl: {torch.sum(return_pred_ctrl.squeeze()*mask_t0)/torch.sum(mask_t0.float())}")
      print(f"Label_return_ctrl: {torch.sum(y0.squeeze()*mask_t0.float())/torch.sum(mask_t0.float())}")
      scheduler.step()
      for name, param in model.t_tower.named_parameters():
        print(f"{name}: mean = {param.data.mean().item():.4f}, std = {param.data.std().item():.4f}")
      # print(f"return_1h_epsilon_grad: {model.return_1h_epsilon.epsilon.grad.item()}")
      # print(f"return_1h_epsilon: {model.return_1h_epsilon.epsilon.item()}")
      

In [6]:
torch.set_printoptions(precision=6)
main()

/Users/bytedance/miniconda3/lib/python3.11/site-packages/torch/utils/data/dataloader.py:617: UserWarning: This DataLoader will create 16 worker processes in total. Our suggested max number of worker in current system is 10 (`cpuset` is not taken into account), which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


----------------Step 1-----------------
Return Trmt Loss: 35.3615
Return Ctrl Loss: 30.0316
Sigma Trmt Loss: 5.1485
Sigma Ctrl Loss: 3.8722
t Loss: 0.7042
return tar Loss: 65.0612
sigma tar Loss: 19.7033
Pred_t: 0.48712292313575745
Label_t: 0.5077881813049316
Pred_return_trmt: 6.31458854675293
Label_return_trmt: 7.9545087814331055
Pred_return_ctrl: 11.0718994140625
Label_return_ctrl: 8.85340404510498
fc1.weight: mean = -0.0065, std = 0.1448
fc1.bias: mean = -0.0014, std = 0.0103
fc2.weight: mean = 0.0017, std = 0.2015
fc2.bias: mean = -0.0018, std = 0.0046
fc3.weight: mean = 0.0031, std = 0.3311
fc3.bias: mean = -0.0022, std = nan


/var/folders/30/cmv9c_5j3mq_kthx63sb1t5c0000gn/T/ipykernel_50963/3025947971.py:75: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /Users/runner/work/_temp/anaconda/conda-bld/pytorch_1729647038473/work/aten/src/ATen/native/ReduceOps.cpp:1823.)
  print(f"{name}: mean = {param.data.mean().item():.4f}, std = {param.data.std().item():.4f}")


----------------Step 2-----------------
Return Trmt Loss: 44.0264
Return Ctrl Loss: 36.9762
Sigma Trmt Loss: 4.9625
Sigma Ctrl Loss: 4.0060
t Loss: 0.6906
return tar Loss: 86.1677
sigma tar Loss: 17.2767
Pred_t: 0.4479326009750366
Label_t: 0.47975078225135803
Pred_return_trmt: 6.936612129211426
Label_return_trmt: 7.633077144622803
Pred_return_ctrl: 10.985363006591797
Label_return_ctrl: 9.276398658752441
fc1.weight: mean = -0.0068, std = 0.1452
fc1.bias: mean = -0.0021, std = 0.0126
fc2.weight: mean = 0.0019, std = 0.2016
fc2.bias: mean = -0.0020, std = 0.0050
fc3.weight: mean = 0.0020, std = 0.3306
fc3.bias: mean = -0.0002, std = nan
----------------Step 3-----------------
Return Trmt Loss: 30.4547
Return Ctrl Loss: 34.4149
Sigma Trmt Loss: 5.5991
Sigma Ctrl Loss: 3.4258
t Loss: 0.6982
return tar Loss: 65.9772
sigma tar Loss: 18.0276
Pred_t: 0.4722291827201843
Label_t: 0.4984423816204071
Pred_return_trmt: 8.384305000305176
Label_return_trmt: 9.807001113891602
Pred_return_ctrl: 10.51749

KeyboardInterrupt: 